# Tail assignment — the four regimes

**Tail assignment** decides which physical aircraft (a *tail*) flies each
scheduled flight. The flights have **fixed** times, so this is
`ortidy.interval_assignment`, not `job_shop`. We walk the four regimes as a
story of increasingly complex operations, then show the `solve_routing`
VRPTW encoding for readers who need retiming or cancellations.

| # | locations | fleet  | operator                       |
|---|-----------|--------|--------------------------------|
| 1 | no        | single | Ryanair (everything from a base) |
| 2 | yes       | single | charter operator, no home base |
| 3 | no        | mixed  | mixed fleet at a base          |
| 4 | yes       | mixed  | mixed fleet, free network      |

In [ ]:
import pandas as pd

import ortidy

## Regime 1 — Ryanair: single fleet, single base

Every flight is an out-and-back from the same base, so location doesn't
matter — only whether two blocks overlap (with a 25-minute turnaround gap).
Nine Dublin blocks; the question is the minimum number of tails.

In [ ]:
blocks = [(385, 580), (400, 725), (420, 870), (615, 810), (835, 1090),
          (840, 1035), (900, 1305), (1065, 1260), (1120, 1290)]
flights = pd.DataFrame({
    "taskId": [f"DUB{i}" for i in range(len(blocks))],
    "start":  [b[0] for b in blocks],
    "end":    [b[1] for b in blocks],
})
r1 = ortidy.interval_assignment(flights, min_gap=25)
print(r1.status, "| tails needed:", r1.objective)
r1.frame.sort_values(["resourceId", "start"])

## Regime 2 — charter operator: single fleet, no base

Now flights criss-cross a network: a tail can only fly the next leg if it's
**at that airport**. Add `start_loc` / `end_loc` and the solver chains legs
into rotations, minimizing tails via a path cover on the compatibility DAG.

In [ ]:
legs = [("A", "B", 600, 700), ("B", "C", 730, 830), ("C", "A", 900, 1000),
        ("A", "C", 620, 750), ("C", "B", 800, 900), ("B", "A", 930, 1030)]
network = pd.DataFrame({
    "taskId":    [f"L{i}" for i in range(len(legs))],
    "start_loc": [f[0] for f in legs],
    "end_loc":   [f[1] for f in legs],
    "start":     [f[2] for f in legs],
    "end":       [f[3] for f in legs],
})
r2 = ortidy.interval_assignment(
    network, start_location_column="start_loc", end_location_column="end_loc",
    min_gap=25,
)
print(r2.status, "| tails needed:", r2.objective)
r2.frame.sort_values(["resourceId", "start"])

## Regime 3 — mixed fleet at a base

Two aircraft types now: short-range **SR** (cheap) and long-range **LR**
(costly). The two longest blocks need an LR. `resources` gives per-type
counts and fixed costs; `eligibility` restricts specific flights. The
objective becomes total fixed cost, and each flight gets a `resourceType`.

In [ ]:
resources = pd.DataFrame({
    "resourceType": ["SR", "LR"],
    "count":        [8, 2],
    "fixed_cost":   [1, 3],
})
# The two longest blocks (DUB2, DUB6) may only fly on an LR.
eligibility = pd.DataFrame({"taskId": ["DUB2", "DUB6"], "resourceType": ["LR", "LR"]})
r3 = ortidy.interval_assignment(
    flights, resources=resources, eligibility=eligibility, min_gap=25,
)
print(r3.status, "| fleet cost:", r3.objective)
r3.frame.sort_values(["resourceType", "resourceId", "start"])

## Regime 4 — mixed fleet, free network

The general case: locations **and** a typed fleet with count caps. One route
(A→C) is LR-only; with SR(count 3, cost 1) and LR(count 1, cost 3) the
optimum is one LR rotation plus one SR rotation — cost 4. Solved as a
per-type flow on the DAG.

In [ ]:
resources4 = pd.DataFrame({
    "resourceType": ["SR", "LR"],
    "count":        [3, 1],
    "fixed_cost":   [1, 3],
})
eligibility4 = pd.DataFrame({"taskId": ["L3"], "resourceType": ["LR"]})  # A->C
r4 = ortidy.interval_assignment(
    network, start_location_column="start_loc", end_location_column="end_loc",
    resources=resources4, eligibility=eligibility4, min_gap=25,
)
print(r4.status, "| fleet cost:", r4.objective)
r4.frame.sort_values(["resourceType", "resourceId", "start"])

## When you need more: retiming & cancellations via `solve_routing`

`interval_assignment` fixes the schedule. If you want to **retime** flights
(widen arrival windows) or allow **cancellations** (drop unprofitable legs),
model the same network as a VRPTW with `solve_routing`: build a
time-distance matrix, add `time_windows` (widen for retiming) and
`penalties` (for optional/cancellable flights).

Below, tidy pandas idioms build the matrix: `merge(how="cross")` +
`.eval()` + `.pivot()`.

In [ ]:
# One node per leg's departure event, keyed by airport + time.
nodes = network.rename(columns={"start_loc": "loc", "start": "t"})[["taskId", "loc", "t"]]
nodes = pd.concat([pd.DataFrame({"taskId": ["DEPOT"], "loc": ["A"], "t": [0]}), nodes],
                  ignore_index=True)
# Cross-join to all node pairs and score travel time (same-airport = wait).
pairs = nodes.merge(nodes, how="cross", suffixes=("_i", "_j"))
pairs = pairs.eval("cost = abs(t_j - t_i)")
matrix = pairs.pivot(index="taskId_i", columns="taskId_j", values="cost")
matrix.head()

In [ ]:
# Widen a window to allow retiming; add a penalty to allow cancelling a leg.
windows = pd.DataFrame({
    "node": nodes["taskId"],
    "open": 0,
    "close": nodes["t"] + 60,     # up to 60 min late
})
penalties = pd.DataFrame({"node": ["L4"], "penalty": [500]})  # L4 is cancellable
routed = ortidy.solve_routing(
    matrix, vehicles=3, starting_point="DEPOT",
    time_windows=windows, penalties=penalties, max_distance=100_000,
)
print(routed.status, "| vehicles used:", routed.metadata["vehicles_used"],
      "| dropped:", routed.metadata["dropped"])

### Scope

`interval_assignment` solves **one** planning period. Multi-day continuity
(tonight's end airport = tomorrow's start), maintenance routing, and initial
tail positions for recovery are out of scope — use `solve_routing` with
per-vehicle start/end locations as the escape hatch when you need them.